# DisasterSeverity — Safe 3-Model Ensemble (targeting 0.82+)

## What went wrong in the 0.68 run
Changed 5 things at once: class weights + FGM→AWP + LLRD + calibration + new model.  
The class-weight 'fix' (sqrt) removed a boost the model actually needed — competition
test sets skew toward harder events, so 161 Catastrophic predictions may have been  
largely correct.

## What this notebook changes (and doesn't)
```
KEPT EXACTLY from 0.784:  class weights, FGM, R-Drop, label smoothing,
                           cosine LR, warmup, 5-fold, pseudo-labelling setup

ONLY NEW THING:           add 2 more base-sized models and score-weight ensemble
```

| Model | Size | Architecture | Why useful |
|-------|------|-------------|------------|
| `csebuetnlp/banglabert` | 110M | ELECTRA | Bangla-specific, your 0.784 baseline |
| `xlm-roberta-base` | 125M | RoBERTa | Multilingual, different arch → different errors |
| `google/muril-base-cased` | 236M | BERT | Google Indic model, covers Bengali |

Three diverse base models ensembled → expected **0.82–0.84**.

In [ ]:
# ── Cell 1: Imports & Seed ─────────────────────────────────────────────────
import os, random, warnings, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from scipy.special import softmax as scipy_softmax
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, set_seed,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [ ]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
#
# ALL TRAINING HYPERS ARE IDENTICAL TO THE 0.784 RUN.
# Only new thing: MODEL_CFGS now has three models.
# ──────────────────────────────────────────────────────────────────────────
BASE_PATH = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"

# Three base-sized models — no 'large' models used
MODEL_CFGS = [
    dict(key="banglabert",  name="csebuetnlp/banglabert",   lr=2e-5, epochs=5),
    dict(key="xlmr_base",   name="xlm-roberta-base",         lr=2e-5, epochs=5),
    dict(key="muril",       name="google/muril-base-cased",  lr=2e-5, epochs=5),
]

# ── Identical to 0.784 baseline ───────────────────────────────────────────
MAX_LEN         = 128
BATCH_SIZE      = 16
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.10
LABEL_SMOOTH    = 0.10
N_FOLDS         = 5
SEED            = 42
RDROP_ALPHA     = 0.50
FGM_EPSILON     = 0.50     # same as baseline
PSEUDO_THRESH   = 0.85     # slightly looser than 0.90 — gets more pseudo samples
PSEUDO_EPOCHS   = 3
PSEUDO_LR       = 1e-5

label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}
NUM_LABELS        = 5

In [ ]:
# ── Cell 3: Data Loading ───────────────────────────────────────────────────
print("Loading data...")
train = pd.read_csv(f"{BASE_PATH}train.csv")
test  = pd.read_csv(f"{BASE_PATH}test.csv")
val   = pd.read_csv(f"{BASE_PATH}validation.csv")

train = pd.concat([train, val]).reset_index(drop=True)

# Category-aware input — same as baseline
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

train["label_id"] = train["label"].map(label_map)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train["fold"] = -1
for fold, (_, vi) in enumerate(skf.split(train, train["label_id"])):
    train.loc[vi, "fold"] = fold

# ── SAME class weights as the 0.784 run (standard formula, NOT sqrt) ──────
raw_counts    = np.bincount(train["label_id"].values, minlength=NUM_LABELS).astype(float)
CLASS_WEIGHTS = (len(train) / (NUM_LABELS * raw_counts)).tolist()
print("Class weights (unchanged from 0.784 baseline):")
for label, w in zip(label_map, CLASS_WEIGHTS):
    print(f"  {label:12s}: {w:.3f}")
print(f"\nTrain: {len(train)} | Test: {len(test)}")

In [ ]:
# ── Cell 4: FGM + Metrics + AdvancedTrainer ────────────────────────────────
#
# THIS IS IDENTICAL TO THE 0.784 TRAINING CODE.
# FGM on embeddings only (NOT AWP). No LLRD. No calibration.
# ──────────────────────────────────────────────────────────────────────────

class FGM:
    """Fast Gradient Method — perturbs word embeddings only."""
    def __init__(self, model):
        self.model  = model
        self.backup = {}

    def attack(self, epsilon=0.5, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    param.data.add_(epsilon * param.grad / norm)

    def restore(self, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds), "f1": f1}


class AdvancedTrainer(Trainer):
    """
    Exact same trainer as the 0.784 run:
      • class-weighted CE + label smoothing
      • R-Drop (symmetric KL between two stochastic forward passes)
      • FGM adversarial embedding perturbation
    """

    def _ce(self, logits, labels):
        wt = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(logits.device)
        return nn.CrossEntropyLoss(weight=wt, label_smoothing=LABEL_SMOOTH)(logits, labels)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        if model.training and not getattr(self, "_fgm_mode", False):
            # R-Drop: two forward passes with different dropout masks
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce   = (self._ce(out1.logits, labels) + self._ce(out2.logits, labels)) / 2
            p1, p2 = F.softmax(out1.logits, -1), F.softmax(out2.logits, -1)
            kl = (F.kl_div(out1.logits.log_softmax(-1), p2, reduction="batchmean") +
                  F.kl_div(out2.logits.log_softmax(-1), p1, reduction="batchmean")) / 2
            loss = ce + RDROP_ALPHA * kl
            return (loss, out1) if return_outputs else loss
        else:
            out  = model(**inputs)
            loss = self._ce(out.logits, labels)
            return (loss, out) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        inputs = self._prepare_inputs(inputs)
        labels = inputs["labels"].clone()

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        scale = self.args.gradient_accumulation_steps
        self.accelerator.backward(loss / scale if scale > 1 else loss)

        # FGM on embeddings (same as 0.784 baseline)
        fgm = FGM(model)
        fgm.attack(epsilon=FGM_EPSILON)
        inputs["labels"] = labels
        self._fgm_mode = True
        with self.compute_loss_context_manager():
            loss_adv = self.compute_loss(model, inputs)
        self._fgm_mode = False
        self.accelerator.backward(loss_adv / scale if scale > 1 else loss_adv)
        fgm.restore()

        return loss.detach()

In [ ]:
# ── Cell 5: Generic Training Function ─────────────────────────────────────
def train_model(model_cfg):
    """
    5-fold CV for one model config.
    Returns (mean test logits, mean CV F1).
    Saves fold logits to disk in case of session timeout.
    """
    key       = model_cfg["key"]
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["name"])

    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length",
                         truncation=True, max_length=MAX_LEN)

    tok_test     = Dataset.from_pandas(test[["text"]]).map(tokenize_fn, batched=True)
    fold_logits  = []
    cv_f1s       = []

    for fold in range(N_FOLDS):
        print(f"\n{'='*16} [{key.upper()}] FOLD {fold+1}/{N_FOLDS} {'='*16}")
        seed_everything(SEED + fold)      # different seed per fold for diversity

        trn_df = train[train["fold"] != fold].reset_index(drop=True)
        val_df = train[train["fold"] == fold].reset_index(drop=True)

        tok_trn = (Dataset.from_pandas(
            trn_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))
        tok_val = (Dataset.from_pandas(
            val_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))

        model = AutoModelForSequenceClassification.from_pretrained(
            model_cfg["name"], num_labels=NUM_LABELS
        )

        # Identical TrainingArguments to 0.784 run
        args = TrainingArguments(
            output_dir                  = f"/kaggle/working/{key}_fold{fold}",
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            learning_rate               = model_cfg["lr"],
            per_device_train_batch_size = BATCH_SIZE,
            per_device_eval_batch_size  = BATCH_SIZE,
            num_train_epochs            = model_cfg["epochs"],
            warmup_ratio                = WARMUP_RATIO,
            lr_scheduler_type           = "cosine",
            weight_decay                = WEIGHT_DECAY,
            load_best_model_at_end      = True,
            metric_for_best_model       = "f1",
            greater_is_better           = True,
            report_to                   = "none",
            save_total_limit            = 1,
        )

        trainer = AdvancedTrainer(
            model=model, args=args,
            train_dataset=tok_trn, eval_dataset=tok_val,
            compute_metrics=compute_metrics,
        )
        trainer.train()

        best_f1 = max(
            (x.get("eval_f1", -1) for x in trainer.state.log_history), default=0
        )
        cv_f1s.append(best_f1)
        print(f"  → Fold {fold+1} best F1: {best_f1:.4f}")

        fold_logits.append(trainer.predict(tok_test).predictions)

    mean_cv = float(np.mean(cv_f1s))
    print(f"\n[{key}] Mean CV F1: {mean_cv:.4f}  folds={[round(s,4) for s in cv_f1s]}")

    arr = np.array(fold_logits)
    np.save(f"/kaggle/working/{key}_logits.npy", arr)

    return arr.mean(axis=0), mean_cv   # (n_test, 5), float

In [ ]:
# ── Cell 6: Train all three models ────────────────────────────────────────
#
# Estimated runtime on T4:
#   banglabert  : ~2 h
#   xlmr_base   : ~2.5 h
#   muril       : ~2.5 h  (larger tokenizer)
#   Total       : ~7 h
#
# To resume after a session timeout:
#   logits, cv = np.load('/kaggle/working/banglabert_logits.npy').mean(axis=0), 0.784
#   all_logits['banglabert'] = logits
#   all_cv['banglabert']     = 0.784
#   then skip directly to Cell 7
# ──────────────────────────────────────────────────────────────────────────
all_logits = {}   # key → (n_test, 5)
all_cv     = {}   # key → float CV F1

for mcfg in MODEL_CFGS:
    seed_everything(SEED)
    logits, cv_f1 = train_model(mcfg)
    all_logits[mcfg["key"]] = logits
    all_cv[mcfg["key"]]     = cv_f1

print("\n── CV Summary ──")
for k in all_cv:
    print(f"  {k}: {all_cv[k]:.4f}")

In [ ]:
# ── Cell 7: Score-weighted ensemble ───────────────────────────────────────
# Weight each model by CV F1 — better CV score → more influence.
# This is safe because BanglaBERT (your proven model) will likely have the
# highest CV and therefore dominate. XLM-R / MuRIL only contribute
# in proportion to how well they actually perform.
total = sum(all_cv.values())
w     = {k: v / total for k, v in all_cv.items()}
print("Ensemble weights:")
for k in sorted(w, key=w.get, reverse=True):
    print(f"  {k}: {w[k]:.3f}  (CV={all_cv[k]:.4f})")

ensemble_logits = sum(w[k] * all_logits[k] for k in all_logits)
final_preds     = np.argmax(ensemble_logits, axis=-1)
test["label"]   = [reverse_label_map[p] for p in final_preds]

# Non-Disaster → always Minimal (same rule as baseline)
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"

print("\nPrediction distribution:")
print(test["label"].value_counts())

In [ ]:
# ── Cell 8: Pseudo-labelling (slightly looser threshold = more samples) ───
print("── Pseudo-Labelling (threshold=0.85) ──")
probs     = scipy_softmax(ensemble_logits, axis=-1)
max_probs = probs.max(axis=-1)
confident = max_probs >= PSEUDO_THRESH
print(f"Confident samples (≥{PSEUDO_THRESH}): {confident.sum()}/{len(test)}")

if confident.sum() > 50:
    pseudo_df             = test[confident].copy()
    pseudo_df["label_id"] = final_preds[confident]
    full_train = pd.concat(
        [train[["text", "label_id"]], pseudo_df[["text", "label_id"]]],
        ignore_index=True,
    )
    print(f"Expanded train: {len(train)} → {len(full_train)} samples")

    # Use BanglaBERT for pseudo training (best-performing, most stable)
    best_key  = max(all_cv, key=all_cv.get)
    best_mcfg = next(m for m in MODEL_CFGS if m["key"] == best_key)
    tok_p     = AutoTokenizer.from_pretrained(best_mcfg["name"])

    def tkn_p(examples):
        return tok_p(examples["text"], padding="max_length",
                     truncation=True, max_length=MAX_LEN)

    tok_full = Dataset.from_pandas(
        full_train.rename(columns={"label_id": "label"})
    ).map(tkn_p, batched=True)
    tok_tst = Dataset.from_pandas(test[["text"]]).map(tkn_p, batched=True)

    pseudo_model = AutoModelForSequenceClassification.from_pretrained(
        best_mcfg["name"], num_labels=NUM_LABELS
    )
    p_args = TrainingArguments(
        output_dir                  = "/kaggle/working/pseudo",
        num_train_epochs            = PSEUDO_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate               = PSEUDO_LR,
        warmup_ratio                = WARMUP_RATIO,
        lr_scheduler_type           = "cosine",
        weight_decay                = WEIGHT_DECAY,
        save_strategy               = "no",
        report_to                   = "none",
    )
    pseudo_trainer = AdvancedTrainer(
        model=pseudo_model, args=p_args,
        train_dataset=tok_full, compute_metrics=compute_metrics,
    )
    pseudo_trainer.train()
    pseudo_logits = pseudo_trainer.predict(tok_tst).predictions

    # Blend: 70% 3-model ensemble + 30% pseudo model
    blended     = 0.70 * ensemble_logits + 0.30 * pseudo_logits
    final_preds = np.argmax(blended, axis=-1)
    test["label"] = [reverse_label_map[p] for p in final_preds]
    test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
    print("Pseudo blend applied. Final distribution:")
    print(test["label"].value_counts())
else:
    print("Too few confident samples, skipping.")

In [ ]:
# ── Cell 9: Submission ─────────────────────────────────────────────────────
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")
print("✅ submission.zip ready")
print(submission["label"].value_counts())
print(submission.head(10))

## Debugging checklist if score drops again

**After Cell 6 completes, check the CV summary:**
- `banglabert` should be ~0.78 (same as your baseline — if it's lower, something changed)
- `xlmr_base` should be 0.73–0.78
- `muril` should be 0.74–0.79

If `banglabert` drops below 0.76, the trainer code has a bug — compare Cell 4 to your working unimodaltest.ipynb line-by-line.

**After Cell 7, check the Catastrophic count:**  
If it stays around 140–180, the model is consistent with the 0.784 run.  
If it drops below 80, something changed the class weights — check Cell 3.

## If you still want to try LLRD or AWP safely
Run them ONE AT A TIME as ablations:
1. First submit this notebook (3-model ensemble baseline)
2. Then add LLRD to `create_optimizer` and re-run only the BanglaBERT fold
3. Compare that fold's val F1 to the version without LLRD
4. Only keep it if val F1 improves